# Lab 6: From scraping books to crawling a link graph
**Information Retrieval & Web Analytics · CADT · Week 6 · 3 hours**  
**Student version — complete the TODO cells, remove their `raise NotImplementedError(...)` lines, and write the Answer prompts.**

In Week 5 you scraped **Books to Scrape** with Beautiful Soup and built the `bookcrawler` Scrapy project. You followed `li.next a` and used `parse_book()` to extract a book's description. Today you keep the same website and extraction selectors, but make scheduling explicit: **seed → frontier → fetch → parse → normalize links → queue unseen URLs → stop**.

| Week 5 skill or artifact | How you extend it today |
|---|---|
| `urljoin(page_url, href)` | Normalize URLs before duplicate detection |
| Two-page pagination loop | A FIFO frontier with depth and request limits |
| `article.product_pod h3 a`, `li.next a` | Discover book pages and the next listing |
| `div.product_main h1`, `#product_description + p` | Reuse detail extraction for the IR corpus |
| `doc_id, title, text, url, category, source` | Keep the same six corpus fields |
| `bookcrawler` and `response.follow()` | Explain Scrapy's scheduler and duplicate filter |
| `ir_books_corpus.jsonl` | Compare today's small sample with last week's export |

**Outcomes:** implement bounded breadth-first crawling; distinguish discovered/queued/attempted/saved pages; explain traps; export a corpus and directed link graph; interpret degrees, reachability and components without confusing missing observations with real dangling pages.

**Pacing (180 min):** reconnect/setup 15 · normalization 20 · extraction 15 · BFS 40 · break 10 · limits/traps 15 · export/QA 20 · graph 25 · reflection 10. Focused crawling, indexing and Scrapy extensions are optional/homework.

**Core boundary:** at most **2 listing URLs**, **6 detail URLs**, **8 HTML request attempts**, depth **2**. Robots is checked separately. Do not crawl all 1,000 books. Internet is required for the live sample; offline exercises still run. The site's prices and ratings are dummy data.

## Part 0. Reconnect with Week 5 (15 min)
Keep this notebook beside your Week 5 files if possible. Your old exports are helpful but **not required**; this notebook includes the imports and helper scaffolds you need.

Install missing packages in the notebook environment only if needed:
```python
%pip install requests beautifulsoup4 pandas matplotlib networkx
```
Do not rerun the live crawl to experiment with limits: later experiments use saved responses. Keep the same two-page boundary as Week 5. Check robots at runtime; a missing robots file does not replace reviewing site terms.

In [ ]:
from pathlib import Path
from collections import deque, Counter, defaultdict
from urllib.parse import urljoin, urldefrag, urlsplit, urlunsplit, parse_qsl, urlencode
from urllib import robotparser
import hashlib, json, re, time, heapq
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

BASE = "https://books.toscrape.com/"
SEED = BASE
USER_AGENT = "CADT-IR-Lab6/1.0 (educational sandbox crawler)"
HEADERS = {"User-Agent": USER_AGENT}
DELAY_SECONDS = 0.5
OUTPUT = Path("lab6_output")
OUTPUT.mkdir(exist_ok=True)
print("Same site as Week 5:", BASE)

### Provided: robots and a cached, polite downloader
Read the helper, then use it unchanged. It checks robots before each HTML fetch; requests have timeouts, delays and no automatic redirects. A redirect must be reviewed rather than silently leaving our scope. A 403 or 429 stops the run. A failed attempt still consumes budget.

`live_fetch()` returns `None` for a robots block, a transport error, an HTTP error, or non-HTML. The BFS function will count every call in `attempted`, so this is a conservative upper bound on actual HTML network requests. `fetch_stats` gives network/error counts. Cached successful responses allow later experiments without new requests.

In [ ]:
# Run once before the live crawl. This cell makes ONE robots request.
robots_response = requests.get(urljoin(BASE, "robots.txt"), headers=HEADERS,
                               timeout=20, allow_redirects=False)
rp = robotparser.RobotFileParser()
if robots_response.status_code == 200:
    rp.parse(robots_response.text.splitlines())
    robots_exists = True
elif robots_response.status_code == 404:
    robots_exists = False
else:
    raise RuntimeError(f"Cannot verify robots rules: HTTP {robots_response.status_code}")

def is_allowed(url):
    return (not robots_exists) or rp.can_fetch(USER_AGENT, url)

effective_delay = max(DELAY_SECONDS, (rp.crawl_delay(USER_AGENT) or 0) if robots_exists else 0)
# Include the robots request in the delay before our next network request.
last_request_end = time.monotonic()
response_cache = {}
fetch_stats = Counter()

def live_fetch(url):
    global last_request_end
    if url in response_cache:
        fetch_stats["cache_hits"] += 1
        return response_cache[url]
    if urlsplit(url)[:2] != urlsplit(BASE)[:2] or not is_allowed(url):
        fetch_stats["scope_or_robots_blocked"] += 1
        return None
    time.sleep(max(0, effective_delay - (time.monotonic() - last_request_end)))
    fetch_stats["network_requests"] += 1
    try:
        response = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=False)
    except requests.RequestException as exc:
        fetch_stats["transport_errors"] += 1
        print(type(exc).__name__, url)
        return None
    finally:
        last_request_end = time.monotonic()
    fetch_stats[f"http_{response.status_code}"] += 1
    if response.status_code in (403, 429):
        raise RuntimeError(f"Stop: HTTP {response.status_code}; review access/Retry-After before any retry.")
    if response.status_code != 200:
        return None
    if "text/html" not in response.headers.get("Content-Type", "").lower():
        fetch_stats["non_html"] += 1
        return None
    response_cache[url] = response
    return response

print("robots HTTP:", robots_response.status_code, "| effective delay:", effective_delay)
print("Seed allowed:", is_allowed(SEED))

**Answer 0.1:** Week 5 already followed links. What new scheduling decisions become visible when you replace its pagination loop with a frontier? Which six fields must today's book corpus preserve?

*Your answer:*

## Part 1. Normalize before checking duplicates (20 min)
Reuse `urljoin`, then remove fragments, lowercase the scheme/host, and remove **only** query keys `utm` and `utm_*`. Keep meaningful keys such as `page`, `id`, `lang`; keep the path's case. Return `None` for non-HTTP(S) URLs or URLs with credentials.

For **this site**, treat `/index.html` as an alias of `/`. Do not collapse a detail path such as `/catalogue/book_123/index.html`. This is an explicit site rule, not a universal rule. Query-bearing URLs are tested below, but are excluded from this lab's live crawl.

In [ ]:
def normalize_url(href, page_url):
    """Canonical absolute URL, or None for unsupported links."""
    # TODO 1: resolve relative href; urldefrag; split; validate scheme/credentials.
    # TODO 2: remove only tracking keys; rebuild query with urlencode(...).
    # TODO 3: lowercase scheme/netloc; empty path -> "/"; root /index.html -> "/".
    raise NotImplementedError("Implement normalize_url")

In [ ]:
tests = [
    ("catalogue/book_1/index.html#reviews", BASE, BASE + "catalogue/book_1/index.html"),
    ("../book_2/index.html?utm_source=nav", BASE + "catalogue/book_1/index.html", BASE + "catalogue/book_2/index.html"),
    ("/catalogue/page-2.html?page=2&utm=x&lang=km", BASE, BASE + "catalogue/page-2.html?page=2&lang=km"),
    ("HTTPS://BOOKS.TOSCRAPE.COM/index.html#top", BASE, BASE),
    ("/catalogue/MixedCase/index.html", BASE, BASE + "catalogue/MixedCase/index.html"),
    ("mailto:reader@example.org", BASE, None),
]
for href, current, expected in tests:
    actual = normalize_url(href, current)
    assert actual == expected, (href, actual, expected)
print("Normalization checks passed.")

# Provided: narrow scope for this exercise, not all same-host URLs.
def page_kind(url):
    if not url:
        return None
    p = urlsplit(url)
    if p.scheme != "https" or p.netloc != "books.toscrape.com" or p.query:
        return None
    if p.path in ("/", "/catalogue/page-2.html"):
        return "listing"
    if re.fullmatch(r"/catalogue/[^/]+/index\.html", p.path):
        return "book"
    return None
assert page_kind(BASE + "catalogue/page-3.html") is None
assert page_kind(BASE + "catalogue/category/books/poetry_23/index.html") is None

**Answer 1.1:** Why is removing every query parameter unsafe? Why do we normalize **before** adding a URL to `seen`? Explain why detail-page `/index.html` must remain in the URL.

*Your answer:*

## Part 2. Reuse Week 5 extraction (15 min)
The downloader gets HTML; these functions only parse it. You may adapt your Week 5 solution. Work from response bytes to let Beautiful Soup read the HTML encoding.

For the teaching graph, record all in-scope `a[href]` links (including breadcrumbs), using visible anchor text as a label. The same selector policy applies to every saved page. For scheduling, use the quotas below; recording an edge does **not** mean requesting its target.

In [ ]:
def extract_links(soup, page_url):
    """Return unique (normalized_target, anchor_text) pairs in HTML order."""
    # TODO 2.1: iterate soup.select("a[href]"); normalize; retain page_kind != None.
    # Deduplicate targets within this source page while keeping first occurrence.
    raise NotImplementedError("Implement extract_links")

def parse_book(soup, url):
    """Week 5-compatible six-field IR document; no HTTP requests here."""
    # TODO 2.2: title from div.product_main h1; description from #product_description + p.
    # Category: ul.breadcrumb li:nth-child(3) a; use "" for a missing field.
    # doc_id = hashlib.sha1(url.encode("utf-8")).hexdigest()[:12]
    # text = title + "\n" + description (strip outer whitespace).
    # Return doc_id, title, text, url, category, source="books.toscrape.com".
    raise NotImplementedError("Adapt Week 5 detail parser")

In [ ]:
# Tiny offline check: missing description is permitted, navigation must not become IR text.
fixture = BeautifulSoup("""<html><div class="product_main"><h1>Sample Book</h1></div>
<ul class="breadcrumb"><li>Home</li><li>Books</li><li><a>Poetry</a></li></ul>
<a href="../../index.html">Home</a><a href="../../index.html#top">Home again</a>
<a href="mailto:help@example.org">Email</a></html>""", "html.parser")
fixture_url = BASE + "catalogue/sample_1/index.html"
doc = parse_book(fixture, fixture_url)
assert doc["title"] == "Sample Book" and doc["text"] == "Sample Book"
assert doc["category"] == "Poetry"
assert len(extract_links(fixture, fixture_url)) == 1
print("Extraction checks passed.")

## Part 3. Build the breadth-first crawler (40 min)
**Run one live crawl only.** FIFO means process URLs in order of discovery, at nondecreasing depth. Enqueue the normalized seed at depth 0 and mark it seen immediately.

Use these separate collections:

| Collection/counter | Meaning |
|---|---|
| `frontier` | queued `(url, depth)` pairs |
| `seen` | URLs admitted to frontier, including seed |
| `attempted` | every call to `fetch`, including unsuccessful calls |
| `pages` | successfully downloaded HTML page snapshots |
| `books` | structured IR documents from successful **book** pages only |
| `edges` | unique source/target links observed in saved HTML |
| `queued_counts` | admissions by page type, including seed |

Two listing admissions and six book admissions bound the whole run. Count admissions **when enqueueing**, so failed requests cannot free slots for extra URLs. `max_requests` counts attempts rather than successes. Robots is handled by `fetch`, outside this loop. The fixed scope includes only `/`, `/catalogue/page-2.html` and direct book paths.

**Algorithm:** dequeue → attempt fetch → skip failure → save page/extract book → record all observed in-scope edges → admit unseen links if depth/type quotas permit → repeat until frontier empty or request budget used. A link rejected by depth is not marked seen, so a later shallower route could admit it. Record links even at the depth boundary, but do not expand beyond it.

In [ ]:
def crawl(seed, fetch, max_requests=8, max_depth=2, max_listings=2, max_books=6):
    """Return pages, edges, books, stats; fetch is live_fetch or a cached replay."""
    if min(max_requests, max_listings) < 1 or min(max_depth, max_books) < 0:
        raise ValueError("Invalid crawl limits")
    seed = normalize_url(seed, seed)
    if page_kind(seed) != "listing":
        raise ValueError("Use an allowed listing seed")
    frontier = deque([(seed, 0)])
    seen = {seed}
    queued_counts = Counter(listing=1, book=0)
    attempted, pages, books, edges = [], {}, {}, set()
    stats = Counter()
    # TODO 3: implement the loop described above.
    # while frontier and len(attempted) < max_requests:
    #   url, depth = frontier.popleft(); attempted.append(url)
    #   response = fetch(url); if None: count failed_or_blocked and continue
    #   soup = BeautifulSoup(response.content, "html.parser")
    #   save pages[url] = {"url":url, "title":..., "depth":depth,
    #                      "kind":page_kind(url), "text":soup.get_text(" ", strip=True)}
    #   if kind == "book": books[url] = parse_book(soup, url)
    #   for target, anchor in extract_links(soup, url):
    #       edges.add((url, target))  # record BEFORE deciding whether to enqueue
    #       if target in seen: count duplicates_avoided; continue
    #       if depth + 1 > max_depth: count depth_skipped; continue
    #       kind = page_kind(target); limit = max_listings if listing else max_books
    #       if queued_counts[kind] >= limit: count quota_skipped; continue
    #       seen.add(target); queued_counts[kind] += 1; frontier.append((target, depth+1))
    raise NotImplementedError("Implement bounded FIFO traversal")
    stats.update(attempts=len(attempted), saved_pages=len(pages), books=len(books),
                 queued=len(seen), frontier_remaining=len(frontier))
    return pages, sorted(edges), books, stats

In [ ]:
# After implementing the TODO, run ONCE. A clean successful run saves 2 listings + 6 books.
CRAWLED_AT = datetime.now(timezone.utc).isoformat()
pages, edges, books, stats = crawl(SEED, live_fetch)
print("Crawl:", dict(stats), "| downloader:", dict(fetch_stats))
assert stats["attempts"] <= 8
assert len(books) <= 6
assert sum(p["kind"] == "listing" for p in pages.values()) <= 2
page_table = pd.DataFrame(list(pages.values()))
if len(page_table):
    display(page_table[["url", "kind", "depth", "title"]])
else:
    print("No HTML saved. Inspect robots/network results before continuing.")

**Answer 3.1:** Trace the first four removals from your frontier. Why can the second listing be processed after several detail pages even though it is also at depth 1?

*Your answer:*

**Answer 3.2:** If a detail request returns 404, does it count against the request budget? Is it a saved page or an IR document? Why might your count be below 8 pages/6 books?

*Your answer:*

## Break (10 min)

## Part 4. Limits and traps without extra live requests (15 min)
Replay only the responses from Part 3. A cache miss means **not observed in this sample**, not a 404. These comparisons cover a small captured subgraph, not the entire site. Never fill a cache miss by calling `live_fetch` here.

In [ ]:
replay_misses = Counter()
def replay_fetch(url):
    if url not in response_cache:
        replay_misses[url] += 1
    return response_cache.get(url)

rows = []
for depth, budget in [(0, 8), (1, 8), (2, 4), (2, 8)]:
    replay_misses.clear()
    p, e, b, s = crawl(SEED, replay_fetch, max_depth=depth, max_requests=budget)
    rows.append({"depth limit": depth, "attempt budget": budget,
                 "attempts": s["attempts"], "saved HTML": len(p), "book docs": len(b),
                 "cache misses": sum(replay_misses.values())})
display(pd.DataFrame(rows))

In [ ]:
# Offline spider trap: every synthetic calendar page exposes a fresh next URL.
# No HTTP requests occur. A seen set cannot stop an infinite chain of UNIQUE URLs.
def calendar_trap(request_budget):
    frontier, seen, trace = deque([1]), {1}, []
    while frontier and len(trace) < request_budget:
        month = frontier.popleft()
        trace.append(f"https://example.invalid/events?month={month}")
        next_month = month + 1
        if next_month not in seen:
            seen.add(next_month)
            frontier.append(next_month)
    return trace, len(frontier)

trace, remaining = calendar_trap(5)
print(*trace, sep="\n")
print("Frontier still contains", remaining, "URL(s) when the budget stops us.")

**Answer 4.1:** Does `seen` stop the synthetic calendar? Explain the difference between a duplicate loop and infinitely many unique URLs. Name two limits that can stop a trap.

*Your answer:*

**Answer 4.2:** Why can cached replay not tell us how many pages are reachable on the whole book site? Which limits apply to book documents versus HTTP attempts?

*Your answer:*

## Part 5. Save and compare the IR corpus (20 min)
Save both crawl evidence and search documents. `pages.jsonl` contains page snapshots (listings **and** details); `edges.csv` records links; `ir_books_corpus.jsonl` contains **only book documents**, with the same six fields as Week 5. Navigation text belongs in the snapshot, not in book search text. All outputs go under `lab6_output/` to preserve last week's files.

In [ ]:
FIELDS = ["doc_id", "title", "text", "url", "category", "source"]
# TODO 5.1: save pages.values() as UTF-8 JSONL to OUTPUT / "pages.jsonl".
# TODO 5.2: save edges as CSV with source,target headers, using pandas or csv.writer.
# TODO 5.3: save only FIELDS from books.values() to OUTPUT / "ir_books_corpus.jsonl".
# TODO 5.4: save crawl_metadata.json with seed, CRAWLED_AT, scope, limits,
#           effective_delay, robots status, stats and fetch_stats.
raise NotImplementedError("Export crawl evidence and corpus")

In [ ]:
corpus_path = OUTPUT / "ir_books_corpus.jsonl"
records = [json.loads(line) for line in corpus_path.read_text(encoding="utf-8").splitlines() if line.strip()]
assert records, "No documents: investigate the crawl instead of fabricating records."
assert all(set(row) == set(FIELDS) for row in records)
assert len({r["doc_id"] for r in records}) == len(records)
assert len({r["url"] for r in records}) == len(records)
print("Documents:", len(records))
print("Empty titles:", sum(not r["title"].strip() for r in records))
print("Empty text:", sum(not r["text"].strip() for r in records))
print("Title-only documents:", sum(r["text"].strip() == r["title"].strip() for r in records))
display(pd.DataFrame(records)[["doc_id", "title", "category"]])

# Optional comparison: update the path if your Week 5 export is elsewhere.
week5_path = Path("ir_books_corpus.jsonl")
if week5_path.exists():
    old = {r["url"]: r for r in map(json.loads, filter(str.strip, week5_path.read_text(encoding="utf-8").splitlines()))}
    shared = [r for r in records if r["url"] in old]
    print("Shared URLs with Week 5:", len(shared))
    display(pd.DataFrame([{"url": r["url"], "same title": r["title"] == old[r["url"]]["title"]} for r in shared]))
else:
    print("Week 5 export absent: core lab still works; compare its six-field schema manually.")

**Answer 5.1:** Week 5's Scrapy crawl aimed for about 40 books; this crawl has at most 6. Why is this expected? If a shared book's title differs, what would you inspect before deciding which parser is wrong?

*Your answer:*

## Part 6. Read the observed link graph (25 min)
A link from page A to page B is directed **A → B**. Build two graphs:

- `G`: every saved source plus every discovered target, even if the target was not downloaded.
- `H`: the subgraph containing only successfully downloaded pages and edges between them.

Degrees count unique page-to-page links, not repeated anchor tags. In-degree measures observed incoming links; out-degree measures observed outgoing links. A highly linked page is a potential **authority**; a hub points to many pages. Observed degrees alone do not establish global importance.

A target-only node has observed out-degree zero because we did not fetch its HTML. Even a fetched page with no eligible links may have links excluded by our scope. Call these **observed sinks**, and avoid claiming they are true Web-wide dangling pages. Components and reachability also describe only the graph you built.

In [ ]:
# TODO 6.1: make nx.DiGraph G; add all pages as nodes, then all edges.
# Add attribute fetched=(url in pages) for each node.
# TODO 6.2: H = G.subgraph(pages).copy()
raise NotImplementedError("Build observed and fetched-page graphs")

In [ ]:
print("Observed graph:", G.number_of_nodes(), "nodes;", G.number_of_edges(), "edges")
print("Fetched-only graph:", H.number_of_nodes(), "nodes;", H.number_of_edges(), "edges")
degree_table = pd.DataFrame([
    {"url": u, "fetched": u in pages, "in_degree": G.in_degree(u), "out_degree": G.out_degree(u)}
    for u in G
])
if not degree_table.empty:
    display(degree_table.sort_values(["in_degree", "url"], ascending=[False, True]).head(8))
    display(degree_table[degree_table.out_degree == 0])
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, column in zip(axes, ["in_degree", "out_degree"]):
        counts = degree_table[column].value_counts().sort_index()
        ax.bar(counts.index, counts.values)
        ax.set(xlabel=column, ylabel="Observed nodes", title=column.replace("_", " "))
    plt.tight_layout()
    plt.show()
if G:
    print("Weak components:", nx.number_weakly_connected_components(G))
    print("Strong components:", nx.number_strongly_connected_components(G))
    print("Reachable from seed (excluding seed):", len(nx.descendants(G, normalize_url(SEED, SEED))))
    pos = nx.spring_layout(G, seed=6)
    colors = ["#2563eb" if u in pages else "#cbd5e1" for u in G]
    plt.figure(figsize=(10, 6))
    nx.draw_networkx(G, pos, node_color=colors, node_size=110, with_labels=False,
                     arrows=True, arrowsize=8, width=0.6)
    plt.title("Observed graph: blue = fetched, grey = discovered only")
    plt.axis("off")
    plt.show()

**Answer 6.1:** Why does the home page often receive many links? Is that evidence it is the best book search result? Connect your answer to breadcrumb/navigation links.

*Your answer:*

**Answer 6.2:** Pick an observed sink. Was it fetched? What can you conclude about its real outgoing links? Why would blindly treating every target-only node as dangling affect later PageRank calculations?

*Your answer:*

**Answer 6.3:** Explain weak versus strong connectivity using your graph. If the whole observed graph forms one weak component, does that prove you collected every page on the website?

*Your answer:*

## Part 7. Reflection and submission (10 min)
Write 4–6 sentences connecting **Week 5 extraction** to **Week 6 discovery**. Explain what the frontier adds, which limit stopped your run, why saved HTML pages differ from IR documents, and one limitation of the graph.

*Your reflection:*

**Submit:**
- [ ] `Lab6_<StudentID>.ipynb` with core TODOs and answers completed.
- [ ] `lab6_output/pages.jsonl`, `edges.csv`, `ir_books_corpus.jsonl`, `crawl_metadata.json`.
- [ ] Crawl counters, replay comparison, corpus QA and graph figures in notebook outputs.
- [ ] Week 5 schema/title comparison (or explain that the old export was unavailable).

The optional extensions below are not required for a clean run of the core notebook. For your team project, extend Week 5's source-and-ethics plan with seed URLs, scope, frontier policy, request/depth limits, and graph sampling limitations. Keep the project's own schedule; this notebook adds no new deadline.

## Optional A. Focused scheduling on cached book pages
Use **only cached detail responses** and a budget of 3 saved book pages for both policies. This isolates a scheduling idea; it is **not** a full focused crawl or a whole-site recall measurement. Frontier candidates are already known from our bounded sample.

Choose a few words likely to occur in the cached titles, such as `poetry`, `light`, `science` or `velvet`. Score candidates using their **URL slug only**, before reading the response. Compare FIFO candidate order with descending score. Label a cached page relevant using its **parsed title**, after selecting it. Relevance here is a deliberately simple proxy; explain why it is imperfect.

TODO: produce policy, sampled books, relevant sampled, harvest rate (= relevant / sampled). Guard empty denominators. If the scores tie or all relevance labels are zero, report that result; do not invent an improvement. Recall is only meaningful relative to the explicitly defined cached candidate set, never the whole website.

In [ ]:
# OPTIONAL — no live_fetch calls.
# TODO: candidates = [u for u in pages if page_kind(u) == "book"]
# TODO: URL-only score; FIFO vs sorted candidates; at most 3 each.
# TODO: use response_cache and parse_book to evaluate selected titles.
# TODO: compare harvest rates and discuss ties/misleading URL clues.

## Optional B. Corpus → inverted index
Load `lab6_output/ir_books_corpus.jsonl` and reuse your earlier course preprocessing and AND search. Index `record["text"]` (already title + description) rather than navigation-filled HTML text.

TODO: map each term to unique sorted `doc_id` postings; report vocabulary size, document frequencies, and three test queries. Apply **exactly the same** preprocessing to documents and queries. State how the small sample limits search coverage.

## Optional C. Extend the real Week 5 Scrapy project
Keep `bookcrawler` and add a new spider named `books_graph` alongside `books.py`. Reuse `parse_book()` selectors and the two-listing boundary; do not replace the project with the old wiki `runspider` exercise.

Map the concepts first:

| Manual crawler | Scrapy project |
|---|---|
| Frontier | scheduler and queued Requests |
| `seen` | request duplicate filter (request fingerprints) |
| Normalization | explicit site-specific URL policy before scheduling |
| Depth | `DEPTH_LIMIT` |
| Delay/robots | `DOWNLOAD_DELAY`, `ROBOTSTXT_OBEY` |
| Saved book dict | item yielded by `parse_book()` |
| Export | feed exporter |

For an optional live run, retain Week 5's settings, use one concurrent request, and enforce **two listing admissions and six detail admissions** in spider code before yielding requests. Keep `DEPTH_LIMIT=2`; use breadth-first scheduler settings if comparing order. Scrapy's default scheduling is not the manual FIFO implementation.

`CLOSESPIDER_PAGECOUNT` counts responses and can overshoot because of requests in flight. It is **not an exact request-attempt budget** and is not equivalent to our loop's `max_requests`. Do not use it as your only boundary. Use a different output filename so Week 5 exports survive.

**Written task (no new crawl required):** explain how the same extraction callback can serve a different scheduling policy, and why Scrapy's duplicate filter cannot infer that arbitrary query parameters are tracking.

**References:** [Books to Scrape](https://books.toscrape.com/) · [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) · [Scrapy settings](https://docs.scrapy.org/en/latest/topics/settings.html) · [Scrapy close-spider extension](https://docs.scrapy.org/en/latest/topics/extensions.html#module-scrapy.extensions.closespider).